# Hoshen–Kopelman in Julia

Run each cell in order. I use the same eight steps and checks as in the Python lesson. Julia arrays start at index 1. Background still has label 0, but I never use 0 as an index into the parent vector. The algorithm needs only Julia Base. IJulia is needed only to run this as a notebook. For installation, use `julia -e 'using Pkg; Pkg.add("IJulia")'` and open Jupyter. I tested the script; notebook-kernel execution is recorded separately in the check report.

## 1. Write a small grid and inspect it

I begin with a small array that I can inspect by eye. A value of 1 identifies an occupied cell; 0 identifies background. Rows run downward and columns run to the right. These values and the labels are dimensionless. I count connections through shared edges, so corner contact alone does not connect two cells.

In [ ]:
mask = Bool[1 0 1; 1 1 1; 0 0 0]
@assert size(mask) == (3, 3)
@assert sum(mask) == 5
println(Int.(mask))

## 2. Find the representative of a label

The first row appears to contain two separate clusters. Later cells connect them. I therefore need a way to record that two labels belong to the same cluster. In the parent list, parent[k] gives the next label to follow. A label that points to itself is the representative. Label 0 is reserved for background. I first test lookup without scanning a grid.

In [ ]:
function find_label(parent, label)
    while parent[label] != label
        label = parent[label]
    end
    return label
end
parent = [1, 1, 2]
@assert find_label(parent, 3) == 1
println("Label lookup passed.")

## 3. Merge two labels

I find the representatives before merging labels. Otherwise I could link the wrong entries or form a cycle in the parent list. I make the larger representative point to the smaller one. This gives predictable labels in the worked example. I do not need a more elaborate data structure for this small lesson.

In [ ]:
function merge_labels!(parent, first, second)
    first_root = find_label(parent, first)
    second_root = find_label(parent, second)
    root = min(first_root, second_root)
    parent[max(first_root, second_root)] = root
    return root
end
parent = [1, 2, 3]
merge_labels!(parent, 2, 3)
merge_labels!(parent, 1, 3)
@assert [find_label(parent, k) for k in 1:3] == [1, 1, 1]
println("Label merging passed.")

## 4. Scan the grid once

I visit rows from top to bottom, and cells within each row from left to right. Only the upper and left neighbors have already been visited. If both are background, I create a label. If one is occupied, I use its label. If both are occupied, I merge their labels. Each shared-edge connection is considered when its later cell is visited. The returned labels are provisional: some different numbers may already refer to the same cluster.

In [ ]:
function scan_grid(mask)
    rows, columns = size(mask)
    labels = zeros(Int, rows, columns)
    parent = Int[]
    for row in 1:rows
        for column in 1:columns
            if !mask[row, column]
                continue
            end
            above = row > 1 ? labels[row - 1, column] : 0
            left = column > 1 ? labels[row, column - 1] : 0
            if above == 0 && left == 0
                new_label = length(parent) + 1
                push!(parent, new_label)
                labels[row, column] = new_label
            elseif above == 0
                labels[row, column] = find_label(parent, left)
            elseif left == 0
                labels[row, column] = find_label(parent, above)
            else
                labels[row, column] = merge_labels!(parent, above, left)
            end
        end
    end
    return labels, parent
end
labels, parent = scan_grid(mask)
@assert labels[1, 1] != labels[1, 3]
@assert find_label(parent, labels[1, 1]) == find_label(parent, labels[1, 3])
println("Provisional labels: ", labels)

## 5. Replace provisional labels and count cells

I now replace each occupied cell label by its representative. I renumber the surviving representatives from 1 upward. The size of a cluster is the number of cells with its final label. For a uniform two-dimensional grid, its area is this count multiplied by the cell area. A count has no units; an area has the units of the squared grid spacing. The largest-cluster fraction is the largest count divided by the occupied-cell count, not the total grid-cell count.

In [ ]:
function finish_labels(labels, parent)
    final = zeros(Int, size(labels))
    numbers = Dict{Int, Int}()
    for row in axes(labels, 1)
        for column in axes(labels, 2)
            label = labels[row, column]
            if label == 0
                continue
            end
            root = find_label(parent, label)
            if !haskey(numbers, root)
                numbers[root] = length(numbers) + 1
            end
            final[row, column] = numbers[root]
        end
    end
    sizes = [count(==(k), final) for k in 1:length(numbers)]
    return final, sizes
end
final, sizes = finish_labels(labels, parent)
@assert sizes == [5]
@assert (final .> 0) == mask
println("Final labels: ", final, "; sizes: ", sizes)

## 6. Add periodic connections as a separate step

I keep the bounded scan unchanged. For periodic data I then connect occupied cells at matching positions on opposite edges. The top cell in a column is adjacent to the bottom cell in that same column. The left and right cells in a row are adjacent in the same way. I merge these labels before producing the final counts. The two-cell example has two bounded clusters and one periodic cluster. It is a small island crossing the displayed seam; it does not form a loop around the whole domain.

In [ ]:
function join_periodic_edges!(labels, parent)
    rows, columns = size(labels)
    for column in 1:columns
        top, bottom = labels[1, column], labels[rows, column]
        if top != 0 && bottom != 0
            merge_labels!(parent, top, bottom)
        end
    end
    for row in 1:rows
        left, right = labels[row, 1], labels[row, columns]
        if left != 0 && right != 0
            merge_labels!(parent, left, right)
        end
    end
end
function hoshen_kopelman(mask; periodic=false)
    @assert ndims(mask) == 2 && minimum(size(mask)) > 0
    labels, parent = scan_grid(Bool.(mask))
    if periodic
        join_periodic_edges!(labels, parent)
    end
    return finish_labels(labels, parent)
end
seam = falses(5, 5)
seam[1, 3] = seam[end, 3] = true
@assert hoshen_kopelman(seam)[2] == [1, 1]
@assert hoshen_kopelman(seam; periodic=true)[2] == [2]
println("Periodic seam check passed.")

## 7. Test examples before analyzing a microstructure

I test an empty grid, a filled grid, a corner contact and a ring. A ring and a filled square each have one connected component. Cluster labeling cannot distinguish their hole counts. I also test a rectangular grid so that accidentally exchanging the row and column dimensions cannot remain hidden.

In [ ]:
@assert hoshen_kopelman(falses(3, 5))[2] == Int[]
@assert hoshen_kopelman(trues(3, 5))[2] == [15]
corner = falses(3, 3)
for k in 1:3
    corner[k, k] = true
end
@assert hoshen_kopelman(corner)[2] == [1, 1, 1]
ring = trues(5, 5)
ring[2:4, 2:4] .= false
@assert hoshen_kopelman(ring)[2] == [16]
rectangle = falses(3, 5)
rectangle[2, 1] = rectangle[2, end] = true
@assert hoshen_kopelman(rectangle)[2] == [1, 1]
@assert hoshen_kopelman(rectangle; periodic=true)[2] == [2]
println("All small-grid checks passed.")

## 8. Apply a threshold and analyze both phases

The composition c and threshold are dimensionless here. I define the high-composition phase using c >= 0.5 and the other phase using c < 0.5. I measure each separately. The occupied fraction uses all grid cells in its denominator. For an absent phase I report a largest-cluster fraction of zero by convention. In a real simulation I would load the saved field instead of this small example. I do not infer winding, holes or numerical convergence from these counts.

In [ ]:
composition = [0.8 0.2 0.7; 0.9 0.6 0.8; 0.1 0.2 0.1]
for (name, phase) in [("High composition", composition .>= 0.5),
                      ("Low composition", composition .< 0.5)]
    phase_labels, phase_sizes = hoshen_kopelman(phase; periodic=true)
    occupied = sum(phase)
    largest_fraction = occupied > 0 ? maximum(phase_sizes) / occupied : 0.0
    @assert sum(phase_sizes) == occupied
    println(name, ": occupied fraction = ", occupied / length(phase),
            "; clusters = ", length(phase_sizes), "; largest fraction = ", largest_fraction)
end